# Modeling
## Studi Kasus: Iris Flower Dataset

---

## 4.1 Pemilihan Model

Pada tahap **Data Preparation** sebelumnya, kita telah melakukan proses standarisasi data (*Scaling*). Karena fitur-fitur sekarang berada dalam skala yang sama (mean ≈ 0, std ≈ 1), sangat rasional bagi kita untuk memilih algoritma yang bekerja berdasarkan perhitungan jarak geometris antar titik data.

Oleh karena itu, pada tahap *Modeling* ini saya memilih algoritma **K-Nearest Neighbors (KNN)**. KNN adalah algoritma klasifikasi *non-parametrik* dan tergolong *lazy learning*, yang artinya ia tidak membangun model matematika yang rumit di awal, melainkan langsung menggunakan jarak Euclidean untuk mencari data terdekat pada fase pengujian. Pemilihan ini sangat relevan untuk dataset Iris karena distribusi fitur per kelasnya mengelompok dengan cukup jelas.

## 4.2 Import Library
Saya mengimpor beberapa modul kunci dari `scikit-learn` untuk pemisahan data, pelatihan model KNN, serta library `joblib` untuk proses penyimpanan model.

In [6]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import joblib

## 4.3 Memuat Dataset Bersih
Saya memuat kembali dataset yang sudah melewati tahap Data Preparation. File `IRIS_after_preparation_for_modeling.csv` ini sudah berisi fitur yang distandarisasi dan label target yang sudah di-encode ke bentuk numerik.

In [7]:
df = pd.read_csv('IRIS_after_preparation_for_modeling.csv')
# Membersihkan baris yang memiliki nilai kosong (NaN) pada target (akibat perbedaan index saat data preparation)
df.dropna(subset=['target'], inplace=True)
df.head()

,sepal_length,sepal_width,petal_length,petal_width,target
0,-0.915509,1.019971,-1.357737,-1.3357,0.0
1,-1.157560,-0.128082,-1.357737,-1.3357,0.0
2,-1.399610,0.331139,-1.414778,-1.3357,0.0
3,-1.520635,0.101529,-1.300696,-1.3357,0.0
4,-1.036535,1.249582,-1.357737,-1.3357,0.0


## 4.4 Pembagian Data Latih dan Data Uji (Data Splitting)

Sebagai *best practice* dalam *machine learning*, kita tidak boleh menguji model dengan data yang sama dengan yang digunakan untuk melatihnya. Oleh karena itu, saya menggunakan `train_test_split` untuk membagi dataset:
- **80% Data Latih (Train)**: Digunakan agar model belajar pola data.
- **20% Data Uji (Test)**: Disembunyikan dan digunakan untuk mengetes akurasi model nanti.

Selain itu, saya menggunakan parameter `stratify=y`. Parameter ini sangat penting agar proporsi dari ketiga spesies Iris (Setosa, Versicolor, Virginica) terbagi secara adil dan merata di dalam set latih maupun set uji.

In [8]:
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Jumlah data latih:', len(X_train))
print('Jumlah data uji:', len(X_test))

Jumlah data latih: 115
Jumlah data uji: 29


## 4.5 Pembangunan Model KNN

Sekarang saatnya membangun model. Saya menetapkan nilai hiperparameter `n_neighbors=5` (K=5). Pemilihan angka 5 ini (yang merupakan angka ganjil) ditujukan untuk menghindari hasil seri (*tie*) saat tetangga-tetangga memberikan "suara" (voting) kelas mana yang dominan. Angka 5 juga cukup moderat, tidak terlalu kecil sehingga sensitif terhadap *noise*, dan tidak terlalu besar sehingga kehilangan detail batas keputusan (*decision boundary*).

In [9]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print('Proses training selesai! Model KNN telah mengenali pola dari data latih.')

Proses training selesai! Model KNN telah mengenali pola dari data latih.


## 4.6 Menyimpan Model dan Data Uji (*Serialization*)

Agar alur kerja (workflow) ini bisa digunakan di *production* tanpa harus melatih ulang model dari nol setiap saat, saya membekukan atau menyimpan (*serialize*) objek model ini ke dalam file `knn_model.pkl` menggunakan `joblib`.

Demikian pula untuk fitur data uji dan label data uji, saya menyimpannya menjadi `.csv` agar nanti di tahap **Evaluation** kita bisa mensimulasikan pengujian secara terpisah.

In [10]:
joblib.dump(knn, 'knn_model.pkl')

# Menyimpan data uji terpisah untuk objektivitas tahap Evaluasi
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
print('Model (.pkl) dan Data Uji (.csv) sukses disimpan untuk tahap selanjutnya!')

Model (.pkl) dan Data Uji (.csv) sukses disimpan untuk tahap selanjutnya!
